In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import patato as pat
from tqdm.auto import tqdm


/Users/telse/Documents/Code/CambridgeBits/MITOX_LW/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
REC_NAME = ("Model Based", "0")
REC_SAVE = "quantification"


In [3]:
def get_table_for_file(f):
    rec = REC_NAME
    f = Path(f)
    pa = pat.PAData.from_hdf5(f)
    if "bicep" not in pa.get_scan_name().lower():
        return None

    SN = pa.scan_reader.file.attrs.get("DeviceSN", "2-22-01")

    if "MITOX" in f.parent.stem and "_Air_" not in pa.get_scan_name():
        return None

    pa.set_default_recon(rec)  # Only extract the Model Based data.

    results = []
    rec = pa.get_scan_reconstructions()

    if not rec:
        print(f"Can't open {f} because file no rec")
        return None

    # Setup spectral unmixing
    unmixer = pat.SpectralUnmixer(
        chromophores=["Hb", "HbO2"],
        rescaling_factor=3,
        wavelengths=pa.get_wavelengths()[pa.get_wavelengths() <= 900],
    )
    thber = pat.THbCalculator()
    so2er = pat.SO2Calculator()

    # Apply spectral unmixing.
    unmixed, _, _ = unmixer.run(rec, None)

    thb, _, _ = thber.run(unmixed, None)
    so2, _, _ = so2er.run(unmixed, None)

    # Loop through ROIs and extract the PA signals.
    for (roi_name, roi_number), roi in pa.get_rois().items():
        if "phantom" in roi_name:
            return None

        # Extract the metadata (ROI name, etc.)
        d = {}
        name, position = roi_name.split("_")
        d["ROI Name"] = name
        d["ROI Position"] = position
        d["ROI Number"] = roi_number
        d["File"] = str(f)
        d["Scan Name"] = pa.get_scan_name()
        d["MITOX ID"] = f.parent.stem
        d["Device S/N"] = SN

        # Extract the mask to apply to the data.
        mask, _ = roi.to_mask_slice(rec)

        # Extract the spectra and mean/median across the regions.
        rd = np.array(rec.raw_data)
        spec = {
            "Wavelengths": pa.get_wavelengths(),
            "Mean": np.nanmean(rd.T[mask.T].T, axis=(0, 2)),
            "Median": np.nanmedian(rd.T[mask.T].T, axis=(0, 2)),
            "Stdev": np.nanstd(rd.T[mask.T].T, axis=(0, 2)),
        }
        for k, v in spec.items():
            d["Spectrum_" + k] = v

        # Extract the positive spectra.
        rd[rd <= 0] = np.nan
        spec = {  # "Wavelengths": pa.get_wavelengths(),
            "Mean": np.nanmean(rd.T[mask.T].T, axis=(0, 2)),
            "Median": np.nanmedian(rd.T[mask.T].T, axis=(0, 2)),
            "Stdev": np.nanstd(rd.T[mask.T].T, axis=(0, 2)),
        }
        for k, v in spec.items():
            d["Spectrum_Positive_" + k] = v

        # Get the maks for the unmixed quantities.
        mask, _ = roi.to_mask_slice(unmixed)

        # Get the unmixed data, apply masks and aggregate.
        umd = np.array(unmixed.raw_data)

        spec = {
            "Mean": np.nanmean(umd.T[mask.T].T, axis=(0, 2)),
            "Median": np.nanmedian(umd.T[mask.T].T, axis=(0, 2)),
            "Stdev": np.nanstd(umd.T[mask.T].T, axis=(0, 2)),
        }

        for k, v in spec.items():
            d["Unmixed_Hb_HbO2_" + k] = v

        # Extract only the Hb/HbO2 with no negatives.
        umd[np.repeat(np.any(umd <= 0, axis=1)[:, None], 2, axis=1)] = np.nan
        spec = {
            "Mean": np.nanmean(umd.T[mask.T].T, axis=(0, 2)),
            "Median": np.nanmedian(umd.T[mask.T].T, axis=(0, 2)),
            "Stdev": np.nanstd(umd.T[mask.T].T, axis=(0, 2)),
        }

        for k, v in spec.items():
            d["Unmixed_Hb_HbO2_Positive_" + k] = v

        # Extract THb, with only positives.
        thbd = np.array(thb.raw_data)
        thbd[np.any(np.isnan(umd), axis=1)[:, None]] = np.nan
        spec = {
            "Mean": np.nanmean(thbd.T[mask.T].T, axis=(0, 2)),
            "Median": np.nanmedian(thbd.T[mask.T].T, axis=(0, 2)),
            "Stdev": np.nanstd(thbd.T[mask.T].T, axis=(0, 2)),
        }
        for k, v in spec.items():
            d["THb_" + k] = v

        # Extract sO2s.
        so2d = np.array(so2.raw_data)
        so2d[np.any(np.isnan(umd), axis=1)[:, None]] = np.nan
        spec = {
            "Mean": np.nanmean(so2d.T[mask.T].T, axis=(0, 2)),
            "Median": np.nanmedian(so2d.T[mask.T].T, axis=(0, 2)),
            "Stdev": np.nanstd(so2d.T[mask.T].T, axis=(0, 2)),
        }
        for k, v in spec.items():
            d["sO2_" + k] = v
        results.append(d)
    return pd.DataFrame(results)


In [4]:
def sortkey(x):
    return int(x.parent.stem[-2:]) if x.parent.stem[-2:].isdigit() else 1000, int(
        x.stem.split("_")[1]
    )


In [5]:
path = "../Data"


In [ ]:
data = Path(path)

files = sorted(data.glob("**/Scan_*.hdf5"), key=sortkey)
dfs = []
for f in tqdm(files):
    tbl = get_table_for_file(f)
    if tbl is not None:
        dfs.append(tbl)


0it [00:00, ?it/s]


In [7]:
df = pd.concat(dfs).reset_index(drop=True)

ValueError: No objects to concatenate

In [8]:
df["MITOX ID"].unique()

NameError: name 'df' is not defined

In [10]:
groups = pd.read_excel("../patient_details/MITOXGroups.xlsx").set_index("Record ID:")
agesex = pd.read_csv("../patient_details/MITOXPAISKINTONEAgeSex.csv").set_index("ID")
itas = pd.read_parquet("../patient_details/ITA_raw.parquet").set_index([
    "SkinID",
    "Region",
])

In [11]:
df["THb_Mean"]

NameError: name 'df' is not defined

In [ ]:
# Make sure that scalar columns are scalar.
columns_to_flatten = [
    f"{um}_{met}" for um in ["THb", "sO2"] for met in ["Mean", "Median", "Stdev"]
]
for c in columns_to_flatten:
    df[c] = df[c].apply(lambda x: x[0])

# Add affected patient/healthy volunteer labels.
df["Group"] = df["MITOX ID"].apply(
    lambda x: (
        "Healthy volunteer"
        if "SKIN" in x
        else "Affected patient"
        if "Study_" in x
        else groups.loc[x, "Group"]
    )
)


In [ ]:
# Get age/sex/etc labels
df = df[df["Scan Name"].apply(lambda x: "bicep" in x.lower())].reset_index(drop=True)
df["Age"] = df["MITOX ID"].apply(lambda x: agesex.loc[x, "Age"])
df["Sex"] = df["MITOX ID"].apply(lambda x: agesex.loc[x, "Sex"])
df["ITA"] = df["MITOX ID"].apply(
    lambda x: (
        itas.loc[(x, "Bicep"), "ITA"].values[0]
        if (x, "Bicep") in itas.index
        else np.nan
    )
)


In [ ]:
## Calculate an interpolated ITA for MITOX study based on 700 nm signal.
df["700nmPositiveMean"] = df.apply(
    lambda x: x["Spectrum_Positive_Mean"][
        np.argmin(np.abs(x["Spectrum_Wavelengths"] - 700))
    ],
    axis=1,
)

df_skin_summary = (
    df[df["ROI Name"] == "skin"].groupby("MITOX ID").mean(numeric_only=True)
)

plt.scatter(df_skin_summary["ITA"], df_skin_summary["700nmPositiveMean"])

isn = pd.isna(df_skin_summary["ITA"])
fit = np.polynomial.Polynomial.fit(
    df_skin_summary["700nmPositiveMean"][~isn], df_skin_summary["ITA"][~isn], 2
)

df_skin_summary.loc[isn, "ITA"] = fit(df_skin_summary[isn]["700nmPositiveMean"])
plt.scatter(
    fit(df_skin_summary[isn]["700nmPositiveMean"]),
    df_skin_summary[isn]["700nmPositiveMean"],
)
plt.gca().invert_yaxis()

df["ITA"] = df["MITOX ID"].apply(lambda x: df_skin_summary.loc[x, "ITA"])


In [ ]:
df.to_pickle(f"{REC_SAVE}/extracted_data.pickle")


In [ ]:
df["Scan Name"].unique()
